# Phase 2.5: Geometry Preprocessing & UV Displacement Dataset Engine

**Objective:** Transform raw high-resolution 3D scans (e.g. FaceScape) into paired 512×512 UV displacement maps, position maps, normal maps, and facial validity masks.

**Hardware Constraint:** This notebook runs entirely on a **CPU-only Kaggle Session (0 GPU quota consumed)**.

**Pipeline Steps:**
1. Ray-casting from neutral FLAME base mesh along outward vertex normals to intersect scan surfaces.
2. Barycentric UV triangle rasterization over FLAME UV layout at 512×512 resolution.
3. Empirical $p_{99}$ measurement across scan corpus.
4. Lossless 16-bit uint PNG storage ($d_{\text{norm}} \in [-1, 1] \to [0, 65535]$).
5. Subject-stratified dataset packaging for Stage 3 Detail GAN training.

In [ ]:
# ── CELL 1: Environment & Dependency Setup (CPU Session) ─────────────────────────
!pip install trimesh opencv-python Pillow pyyaml tqdm --quiet

import trimesh
import cv2
import numpy as np
from pathlib import Path
import json

print(f"Trimesh version: {trimesh.__version__}")
print(f"OpenCV version:  {cv2.__version__}")

In [ ]:
# ── CELL 2: Discover Attached Raw Scans & FLAME Model Assets ────────────────────
SCAN_DIR = Path('/kaggle/input/facescape-raw-scans')
if not SCAN_DIR.exists():
    SCAN_DIR = Path('./data/raw_scans')

FLAME_PATH = Path('/kaggle/input/flame-model/generic_model.pkl')
if not FLAME_PATH.exists():
    FLAME_PATH = Path('./data/flame_model/generic_model.pkl')

OUTPUT_DIR = Path('/kaggle/working/uv_displacement_dataset_512')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw Scans Directory: {SCAN_DIR.resolve()} (Exists: {SCAN_DIR.exists()})")
print(f"FLAME Model Path:    {FLAME_PATH.resolve()} (Exists: {FLAME_PATH.exists()})")
print(f"Output Dataset Path: {OUTPUT_DIR.resolve()}")

In [ ]:
# ── CELL 3: Run Geometry Preprocessing Engine ──────────────────────────────────
!python scripts/build_uv_displacement_dataset.py \
    --scan_dir {SCAN_DIR} \
    --flame_model {FLAME_PATH} \
    --output_dir {OUTPUT_DIR} \
    --resolution 512

In [ ]:
# ── CELL 4: Verify Normalization Statistics & Lossless Round-Trip ───────────────
stats_file = OUTPUT_DIR / 'normalization_stats.json'
if stats_file.exists():
    with open(stats_file) as f:
        stats = json.load(f)
    print("\n--- Measured Normalization Statistics ---")
    print(f"Displacement p99: {stats.get('p99_mm', 'N/A')} mm")
    print(f"Resolution:       {stats.get('resolution', 512)}x{stats.get('resolution', 512)}")
    print(f"Sample Count:     {stats.get('num_samples', 0)}")
    print(f"Storage Format:   {stats.get('storage_format')}")
else:
    print(f"Statistics file not found at: {stats_file}")

In [ ]:
# ── CELL 5: Visualize Preprocessed UV Maps (Displacement, Normal, Mask) ────────
import matplotlib.pyplot as plt
from scripts.build_uv_displacement_dataset import decode_displacement_16bit

disp_files = sorted(list(OUTPUT_DIR.glob("*_disp.png")))
if disp_files:
    sample_file = disp_files[0]
    print(f"Visualizing sample: {sample_file.name}")
    u16_disp = cv2.imread(str(sample_file), cv2.IMREAD_UNCHANGED)
    norm_disp = decode_displacement_16bit(u16_disp)

    plt.figure(figsize=(6, 6))
    plt.imshow(norm_disp, cmap='inferno', vmin=-1.0, vmax=1.0)
    plt.title(f"{sample_file.stem} (Normalized Displacement [-1, 1])")
    plt.colorbar(label="Normalized Magnitude")
    plt.axis('off')
    plt.show()
else:
    print("No preprocessed displacement maps found to display.")